# Fase 4 v2 — 05: homografía por líneas con gate de overlap

Arregla las líneas **chuecas que se quedaban fijas**. Cada frame:
1. `field_white_lines` — líneas blancas por color HSV **dentro** del `green_floor` de SAM3.
2. `solve_lines_masks` — ajusta esquinas (extrapoladas, tolera oclusión), orienta con
   `yellow_zone`/`blue_zone`, elige la H de **mayor overlap** sobre la blanca.
3. `VideoHomographyLines` — re-ajuste **dinámico** cada frame + EMA (estable) + **gate**:
   solo acepta si el overlap supera el umbral → nunca fija algo torcido; si el frame
   ajusta mal, conserva la última H buena.

Cenital: ~82/90 `fit`, overlap ~0.55, círculo central clavado. Lateral: la extracción
de blanco es perfecta pero el ajuste de esquinas falla bajo perspectiva fuerte
(pendiente: detector de líneas Hough/LSD propio).

In [ ]:
import sys, os, subprocess
from pathlib import Path
import numpy as np, cv2, decord

REPO = Path.cwd().resolve()
while not (REPO / 'src/core/field_template.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from src.core.sam3_loader import load_sam3
from src.core.segmentation import detect_classes_in_frame, _load_classes
from src.core.minimap_pipeline import _largest_mask, _largest_component
from src.core.homography import mask_centroid
from src.core import field_landmarks as fl
from src.core.homography_multifeature import VideoHomographyLines

OUT = REPO / 'notebooks/fase_4_homografia/v2_robust/outputs/videos'
OUT.mkdir(parents=True, exist_ok=True)
bundle = load_sam3()
CLS = [c for c in _load_classes() if c['name'] in ('green_floor','yellow_zone','blue_zone')]
print('REPO:', REPO)

In [ ]:
TH = np.linspace(0,2*np.pi,60)
CIRC = np.stack([fl.CENTER_CIRCLE[0]+fl.CENTER_CIRCLE[2]*np.cos(TH), fl.CENTER_CIRCLE[1]+fl.CENTER_CIRCLE[2]*np.sin(TH)],1)
def c2i(Hi,p): return cv2.perspectiveTransform(np.asarray(p,np.float64).reshape(-1,1,2),Hi).reshape(-1,2)
def draw(bgr,H,status,ov):
    img=bgr.copy(); col=(0,255,0) if status=='fit' else (0,200,255)
    cv2.putText(img,status+' ov='+format(ov,'.2f'),(20,46),cv2.FONT_HERSHEY_SIMPLEX,1.1,col,3,cv2.LINE_AA)
    if H is None: return img
    Hi=np.linalg.inv(H); rect=[fl.LANDMARK_POINTS[n] for n in ['inner_tl','inner_tr','inner_br','inner_bl']]
    cv2.polylines(img,[c2i(Hi,rect).astype(np.int32)],True,(0,255,0),2,cv2.LINE_AA)
    cl=c2i(Hi,[fl.LANDMARK_POINTS['center_top'],fl.LANDMARK_POINTS['center_bot']]).astype(np.int32)
    cv2.line(img,tuple(cl[0]),tuple(cl[1]),(0,255,255),2,cv2.LINE_AA)
    cv2.polylines(img,[c2i(Hi,CIRC).astype(np.int32)],True,(255,128,0),2,cv2.LINE_AA); return img

def field_inputs(rgb):
    dets=detect_classes_in_frame(rgb,CLS,bundle); cm=_largest_mask(dets.get('green_floor',[]))
    if cm is None: return None,None,None
    cm=_largest_component(cm); ym=_largest_mask(dets.get('yellow_zone',[])); bm=_largest_mask(dets.get('blue_zone',[]))
    return cm,(mask_centroid(ym) if ym is not None else None),(mask_centroid(bm) if bm is not None else None)

def render(name, path, n=90, step=3, ow=960, fps=12):
    vr=decord.VideoReader(path); idxs=list(range(0,len(vr),step))[:n]; vh=VideoHomographyLines()
    H0,W0=vr[idxs[0]].asnumpy().shape[:2]; sc=ow/W0; oh=int(H0*sc)
    tmp=str(OUT/f'{name}_tmp.mp4'); wr=cv2.VideoWriter(tmp,cv2.VideoWriter_fourcc(*'mp4v'),fps,(ow,oh))
    for i in idxs:
        rgb=vr[i].asnumpy(); bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
        cm,yc,bc=field_inputs(rgb)
        H,st,ov=(vh.H,'kept',0.0) if cm is None else vh.update(bgr,cm,yc,bc)
        wr.write(cv2.resize(draw(bgr,H,st,ov),(ow,oh)))
    wr.release(); fin=str(OUT/f'{name}_lines.mp4')
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',tmp,'-vcodec','libx264','-pix_fmt','yuv420p',fin],check=True); os.remove(tmp)
    print(name, vh.stats(), 'MB', round(os.path.getsize(fin)/1e6,2)); return fin

render('cenital_9933','/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9933.MOV')

## Pendiente lateral

`solve_lines_masks` usa `inner_corners_extrapolated` (minAreaRect + 4 lados), que bajo
perspectiva lateral fuerte + arcos de área se corrompe (overlap ~0.31). Siguiente:
detector de líneas por segmentos (Hough/LSD) → agrupar las 4 líneas de campo reales →
correspondencias punto+línea (estilo PnLCalib) para registrar lateral con precisión.